# Process to make the 0th order consensus

We want to make a consensus training dataset, by running extractions on the 1000 sample images using the models as downloaded, and then making a set of consensus transcriptions - where we judge the accuracy of each transcription by whether we get agreement between the models.


## 1. Model checkpoints to use

We will use 5 different models as-downloaded

- smolvlm2 - HuggingFaceTB/SmolVLM2-2.2B-Instruct
- granite4 - ibm-granite/granite-vision-4.1-4b
- gemma3 - google/gemma-3-4b-it
- gemma4 - google/gemma-4-E4B-it
- ministral - mistralai/Mistral-Small-3.1-24B-Instruct-2503

We will use the set of 1000 images drawn as a random sample from the full image set.

- documents/DailyRainfall_UK/consensus_1000/images


## 2. Run the extractions

Each command submits one extraction job to Azure. Run them in the terminal. They will complete in about 2.5 hours (+queuing time).

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model smolvlm2 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model granite4 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model gemma3 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model gemma4 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_0 --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --model ministral --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions_0 --total-shards 4 --batch-size 15 --extraction-registry outputs/extraction_registry.json extract 
```


## 4. Download the extractions from Azure

Get the extraction run_names from the extraction registry. In this case, they were:

- SmolVLM: 20260615-160250
- Granite4: 20260615-161844
- Gemma3: 20260615-161905
- Gemma4: 20260615-161920
- Ministral: 20260615-161934

Then download them in turn

```bash
bash scripts/aml_download.sh --run-name 20260615-160250
bash scripts/aml_download.sh --run-name 20260615-161844
bash scripts/aml_download.sh --run-name 20260615-161905
bash scripts/aml_download.sh --run-name 20260615-161920
bash scripts/aml_download.sh --run-name 20260615-161934
```




## 5. Make consensus from the downloaded extractions

Looks for cases where multiple extractions agree and flags those sections as reliable.

```bash
bash scripts/run_consensus_pipeline.sh --dataset-root outputs/consensus_dataset_1000 --variant-name consensus_0th_order --threshold 3 --null-threshold 5 --precision 3 --extraction-dir outputs/extractions/20260615-160250 --extraction-dir outputs/extractions/20260615-161844 --extraction-dir outputs/extractions/20260615-161905 --extraction-dir outputs/extractions/20260615-161920 --extraction-dir outputs/extractions/20260615-161934 --validate --validation-sample-denominator 20
```

This will produce a consensus in ```outputs/consensus_dataset_1000/consensus_0th_order``` with transcriptions (ready to use for further fine-tuning), summary and validation figures for 1/20 of the cases.